# 07 · Comprehensions, iterators and generators

After this notebook you can write clear comprehensions, explain the iterator protocol, build lazy generator pipelines that process data far bigger than memory, batch texts for an embeddings API with `itertools`, and explain why streaming LLM output is just a generator you loop over.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [06 · Functions](06_functions_args_scope_lambda.ipynb)

## Why this matters in interviews

- "Process a file bigger than RAM" is a standard data-engineering question, and the idiomatic Python answer is a generator pipeline — see the bank answers below.
- Comprehension fluency is assumed in coding rounds; knowing when a generator expression is better (memory) and when a plain loop is better (readability) is the senior version.
- Streaming LLM output, batching for embeddings, paginating APIs, chunking documents: all are iterators. The single-use (exhaustion) gotcha causes real, silent bugs.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `dt04` | What is a generator in Python and why does it matter here? |
| `dt01` | You need to process a 5 GB CSV on a server with 2 GB of RAM. What do you do? |
| `dt03` | "Streaming" means at least four different things. Which do you mean? |

## What interviewers ask

- List comprehension vs generator expression — what is the difference, and when does it matter?
- What is the iterator protocol? What is the difference between an iterable and an iterator?
- What does `yield` do? How is it different from `return`? What is `yield from`?
- Why does looping over a generator a second time produce nothing?
- How would you batch 10,000 texts for an embeddings API?
- What does `itertools.groupby` require of its input?

## 1 · List comprehensions

`[expression for item in iterable if condition]` builds a list in one readable line. It is equivalent to a `for` loop with `append`, usually a little faster, and its loop variable does not leak into the surrounding scope.

Two shapes people mix up:
- **filter** — the `if` goes at the **end**: `[x for x in xs if x > 0]` drops items;
- **transform** — a conditional *expression* goes at the **front**: `[x if x > 0 else 0 for x in xs]` keeps every item.

**Predict, then run:** how long is each result?

In [ ]:
scores = [0.9, -0.2, 0.4, -0.7, 0.1]
kept = [s for s in scores if s > 0]              # filter
clipped = [s if s > 0 else 0.0 for s in scores]  # transform
print("filter   :", kept)
print("transform:", clipped)

squares_loop = []
for n in range(5):
    squares_loop.append(n * n)
print("loop == comprehension:", squares_loop == [n * n for n in range(5)])
assert len(kept) == 3 and len(clipped) == 5

### Nested comprehensions — and where to stop

Several `for` clauses run in the **same order as nested loops** written top to bottom. One or two clauses read well; beyond that, or with complicated conditions, write the loop.

In [ ]:
docs = ["the cat sat", "the dog ran far"]
tokens = [tok for doc in docs for tok in doc.split()]          # for doc: for tok:
print("flattened:", tokens)
matrix = [[1, 2, 3], [4, 5, 6]]
print("transpose:", [[row[i] for row in matrix] for i in range(3)])
pairs = [(q, d) for q in ["q1", "q2"] for d in ["d1", "d2"] if (q, d) != ("q2", "d1")]
print("pairs    :", pairs)
# Correct, but too dense — a loop with named steps would be kinder to the reader:
dense = [w.upper() for doc in docs if "dog" in doc for w in doc.split() if len(w) > 2 if w != "far"]
print("dense    :", dense)
assert tokens == ["the", "cat", "sat", "the", "dog", "ran", "far"] and dense == ["THE", "DOG", "RAN"]

## 2 · Dict and set comprehensions

The same syntax with braces: `{key: value for …}` and `{value for …}`. A tokenizer vocabulary is a textbook example: every unique token gets an integer id, and the inverse dict maps ids back to tokens.

In [ ]:
corpus = ["the cat sat on the mat", "the dog sat"]
unique = sorted({tok for line in corpus for tok in line.split()})     # set comprehension, sorted for stable ids
vocab = {tok: i for i, tok in enumerate(unique)}                      # token -> id
inverse = {i: tok for tok, i in vocab.items()}                        # id -> token
encoded = [vocab[t] for t in "the dog sat on the mat".split()]
print("vocab  :", vocab)
print("encoded:", encoded, "-> decoded:", " ".join(inverse[i] for i in encoded))
assert " ".join(inverse[i] for i in encoded) == "the dog sat on the mat"

## 3 · Generator expressions: the lazy version

Round brackets instead of square ones — `(x * x for x in data)` — give a **generator expression**. It builds nothing up front and produces one value each time it is asked, so its memory stays constant however many items flow through. When the result is consumed once — by `sum`, `max`, `any`, `"".join`, a `for` loop — a generator expression does the same job without materialising the list. When it is the only argument, drop the extra brackets: `sum(x * x for x in data)`.

(`sys.getsizeof` is shallow, so the list's number below counts only its array of pointers; the int objects add more. Section 7 measures true peak memory with `tracemalloc`. Note that for a handful of items the list is actually *smaller* — a generator has a fixed ~200-byte cost — so laziness pays off for big or unbounded data, not for ten items.)

In [ ]:
import sys

import matplotlib.pyplot as plt

sizes = [10, 1_000, 100_000, 1_000_000]
list_bytes = [sys.getsizeof([n for n in range(k)]) for k in sizes]
gen_bytes = [sys.getsizeof((n for n in range(k))) for k in sizes]
for k, lb, gb in zip(sizes, list_bytes, gen_bytes):
    print(f"{k:>9,} items: list {lb:>10,} bytes | generator {gb:>4} bytes")

fig, ax = plt.subplots(figsize=(8, 3.4))
x = range(len(sizes))
ax.bar([i - 0.2 for i in x], list_bytes, 0.4, label="list comprehension [ ... ]")
ax.bar([i + 0.2 for i in x], gen_bytes, 0.4, label="generator expression ( ... )")
ax.set_yscale("log"); ax.set_xticks(list(x), [f"{k:,}" for k in sizes])
ax.set_title("sys.getsizeof: a list grows with the number of items, a generator does not")
ax.set_xlabel("number of items"); ax.set_ylabel("bytes (log scale)"); ax.legend()
plt.show()
assert len(set(gen_bytes)) == 1 and list_bytes[-1] > 1000 * gen_bytes[-1]
assert sum(x * x for x in range(1000)) == sum([x * x for x in range(1000)])

## 4 · The iterator protocol

Two roles:

- An **iterable** can produce an iterator: it has `__iter__` (lists, strings, dicts, files, ranges, generators).
- An **iterator** produces values one at a time: `__next__` returns the next value or raises `StopIteration` when done, and its `__iter__` returns itself.

`iter(x)` asks an iterable for an iterator; `next(it)` pulls one value; `next(it, default)` returns `default` instead of raising. A `for` loop is this protocol wrapped in a `while True`.

In [ ]:
messages = ["hi", "how are you?", "bye"]
it = iter(messages)
print(next(it), "|", next(it), "|", next(it), "|", next(it, "<exhausted>"))
try:
    next(it)
except StopIteration:
    print("StopIteration raised")

it = iter(messages)                      # what `for m in messages: print(m)` does under the hood
while True:
    try:
        m = next(it)
    except StopIteration:
        break
    print("loop body:", m)
print("a list is an iterator?", hasattr(messages, "__next__"), "| iter(it) is it:", iter(it) is it)
assert next(iter(messages)) == "hi" and next(it, None) is None

### A custom iterator: overlapping chunks for RAG

Implementing the protocol yourself takes `__iter__` and `__next__`. The class below walks a list of tokens in overlapping windows — the classic fixed-size chunking with overlap applied to documents before embedding ([chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb)).

In [ ]:
class ChunkIterator:
    """Yield windows of `size` tokens, each overlapping the previous one by `overlap` tokens."""

    def __init__(self, tokens, size, overlap):
        if not 0 <= overlap < size:
            raise ValueError("need 0 <= overlap < size")
        self.tokens, self.size, self.step = tokens, size, size - overlap
        self.start, self.done = 0, False

    def __iter__(self):
        return self

    def __next__(self):
        if self.done or self.start >= len(self.tokens):
            raise StopIteration
        chunk = self.tokens[self.start:self.start + self.size]
        self.done = self.start + self.size >= len(self.tokens)     # this window reached the end
        self.start += self.step
        return chunk

words = "retrieval augmented generation grounds answers in documents fetched at query time".split()
chunks = list(ChunkIterator(words, size=4, overlap=1))
for c in chunks:
    print(c)
assert all(a[-1:] == b[:1] for a, b in zip(chunks, chunks[1:]))       # consecutive chunks share 1 token
assert chunks[-1][-1] == "time" and len(chunks) == 4

## 5 · Generator functions: `yield`

Writing `__next__` by hand is clumsy. A function containing `yield` is a **generator function**: calling it runs *nothing* — it returns a generator object. Each `next()` runs the body up to the next `yield`, hands that value out, and **pauses** with all its local variables intact; the following `next()` resumes right there. When the function returns, the generator raises `StopIteration`. The chunker above shrinks to five lines.

**Predict, then run:** in what order do the lines print?

In [ ]:
def countdown(n):
    print("  (generator body starts)")
    while n > 0:
        yield n
        n -= 1
    print("  (generator body finishes)")

gen = countdown(2)
print("created:", type(gen).__name__)
print("first next  ->", next(gen))
print("second next ->", next(gen))
print("the rest    ->", list(gen))

def chunk_tokens(tokens, size, overlap):
    step = size - overlap
    for start in range(0, len(tokens), step):
        yield tokens[start:start + size]
        if start + size >= len(tokens):
            return
assert list(chunk_tokens(words, 4, 1)) == chunks

### `yield from`: delegate to another iterable

`yield from iterable` yields every item of another iterable or generator — handy for walking recursive structures and for chaining several sources.

In [ ]:
def walk(node):
    """Yield every string in a nested structure of dicts and lists."""
    if isinstance(node, str):
        yield node
    elif isinstance(node, dict):
        for value in node.values():
            yield from walk(value)
    elif isinstance(node, list):
        for item in node:
            yield from walk(item)

def all_sources():
    yield from ["faq.md", "policy.md"]
    yield from (f"ticket-{i}.txt" for i in range(2))

nested = {"title": "FAQ", "sections": [{"q": "Refunds?", "a": ["30 days", "with receipt"]}]}
print(list(walk(nested)))
print(list(all_sources()))
assert list(walk(nested)) == ["FAQ", "Refunds?", "30 days", "with receipt"] and len(list(all_sources())) == 4

## 6 · The exhaustion gotcha

An iterator is **single-use**. Once consumed it stays empty — and looping over it again raises no error, it just silently does nothing. That applies to generators, `map`, `filter`, `zip`, open files and `csv.reader`. Even `in` consumes the iterator up to the match. If you need two passes, re-create the generator (call the function again) or materialise it with `list()` deliberately — accepting the memory cost.

**Predict, then run:** what do the second `sum` and the final `list` print?

In [ ]:
squares = (x * x for x in range(4))
print("first sum :", sum(squares))
print("second sum:", sum(squares))

lengths = map(len, ["alpha", "be", "gamma"])
print("max(lengths):", max(lengths), "| then list(lengths):", list(lengths))

tokens_it = iter(["a", "b", "c", "d"])
print("'b' in tokens_it:", "b" in tokens_it, "| what is left:", list(tokens_it))

def squares_of(n):                   # re-creatable: each call returns a fresh generator
    return (x * x for x in range(n))
print("two fresh passes:", sum(squares_of(4)), sum(squares_of(4)))
assert sum(squares) == 0 and sum(squares_of(4)) == 14

## 7 · Lazy pipelines for bigger-than-memory data (`dt01`, `dt04`)

Chain generators and each stage pulls one item at a time from the stage before; nothing is materialised, so memory stays flat however big the input. This is the idiomatic Python answer to "the file is bigger than RAM": open the file, stream the rows, reduce as you go — memory is bounded by the size of the *result* (here, one total per model), not by the number of rows.

Below: write a 50,000-row CSV of fake request logs to a temporary folder, then total the tokens per model two ways — load the rows into a list first, or stream them — for growing prefixes of the file (`islice` takes the first *n* rows; more in section 8). `tracemalloc` measures true **peak** memory, counting every object, unlike the shallow `sys.getsizeof`.

In [ ]:
import csv
import random
import tempfile
import tracemalloc
from collections import defaultdict
from itertools import islice
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
log_path = tmp / "requests.csv"
rng = random.Random(0)
with open(log_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["model", "prompt_tokens", "completion_tokens"])
    for _ in range(50_000):
        writer.writerow([rng.choice(["small", "large", "embed"]), rng.randint(10, 2000), rng.randint(0, 500)])
print(f"wrote {log_path.name}: {log_path.stat().st_size / 1e6:.1f} MB")

def totals_eager(path, n_rows):
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(islice(csv.DictReader(f), n_rows))       # every row, as a dict, in memory at once
    totals = defaultdict(int)
    for row in rows:
        totals[row["model"]] += int(row["prompt_tokens"]) + int(row["completion_tokens"])
    return dict(totals)

def totals_streaming(path, n_rows):
    totals = defaultdict(int)                                # bounded by the number of KEYS, not rows
    with open(path, newline="", encoding="utf-8") as f:
        for row in islice(csv.DictReader(f), n_rows):        # one row at a time
            totals[row["model"]] += int(row["prompt_tokens"]) + int(row["completion_tokens"])
    return dict(totals)

def peak_mb(fn, *args):
    tracemalloc.start()
    result = fn(*args)
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return result, peak / 1e6

row_counts = [5_000, 10_000, 20_000, 40_000]
eager_mb, stream_mb = [], []
for n in row_counts:
    eager, e_mb = peak_mb(totals_eager, log_path, n)
    stream, s_mb = peak_mb(totals_streaming, log_path, n)
    assert eager == stream                                  # same answer either way
    eager_mb.append(e_mb); stream_mb.append(s_mb)
print(f"at {row_counts[-1]:,} rows: eager peak {eager_mb[-1]:.1f} MB vs streaming peak {stream_mb[-1]:.2f} MB")
assert eager_mb[-1] > 10 * stream_mb[-1] and stream_mb[-1] < 2 * stream_mb[0] + 0.1

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(row_counts, eager_mb, marker="o", label="list(csv.DictReader(f)) then loop")
ax.plot(row_counts, stream_mb, marker="s", label="for row in csv.DictReader(f)")
ax.set_title("Peak memory while totalling tokens per model")
ax.set_xlabel("rows processed"); ax.set_ylabel("peak memory (MB, tracemalloc)"); ax.legend()
plt.show()

### A pipeline of generator stages

Each stage is a small generator and the final consumer drives everything. Instrumenting the source proves the laziness: building the pipeline reads nothing, and taking the first batch reads only as many lines as that batch needs. `itertools.batched` (Python 3.12+) groups items into tuples of up to *n*.

In [ ]:
from itertools import batched

docs_path = tmp / "docs.txt"
docs_path.write_text("\n".join(
    ["  Refunds take 5 days.  ", "", "SHIPPING is free over $50.", "   ", "Support: 24/7 chat."] * 2_000), encoding="utf-8")

lines_read = 0
def read_lines(path):
    global lines_read
    with open(path, encoding="utf-8") as f:
        for line in f:
            lines_read += 1
            yield line

def clean(lines):
    for line in lines:
        yield " ".join(line.split()).lower()

def non_empty(lines):
    return (line for line in lines if line)

pipeline = batched(non_empty(clean(read_lines(docs_path))), 64)
print("lines read after building the pipeline:", lines_read)
first_batch = next(pipeline)
print("lines read after the first batch      :", lines_read, "| batch size:", len(first_batch))
print("first items:", first_batch[:3])
remaining_batches = sum(1 for _ in pipeline)
print("lines read after draining it          :", lines_read, "| batches:", 1 + remaining_batches)
assert first_batch[0] == "refunds take 5 days." and lines_read == 10_000 and 1 + remaining_batches == 94

## 8 · `itertools`: the iterator toolbox

| Function | Does | AI-engineering use |
|---|---|---|
| `islice(it, n)` | the first `n` items (or a slice) of any iterator | the first k results of a huge or infinite stream |
| `chain(a, b)`, `chain.from_iterable(xs)` | concatenate iterables lazily | merge chunks from several documents |
| `batched(it, n)` (3.12+) | tuples of up to `n` items | batch texts for an embeddings API |
| `groupby(it, key)` | runs of *consecutive* items with equal key — **sort first** | group sorted log rows by model |
| `count(start)` | `start, start + 1, …` forever | ids, attempt numbers |
| `cycle(it)` | repeat an iterable forever | round-robin across API keys or replicas |
| `accumulate(it)` | running totals | cumulative tokens: where does the context budget run out? |
| `pairwise(it)` (3.10+) | consecutive pairs | gaps between timestamps |

In [ ]:
from itertools import accumulate, chain, count, cycle, groupby, pairwise

print("islice(count(100), 3)  :", list(islice(count(100), 3)))
print("chain                  :", list(chain(["a1", "a2"], ["b1"])))
print("chain.from_iterable    :", list(chain.from_iterable([["c1"], ["c2", "c3"]])))
print("batched(range(7), 3)   :", list(batched(range(7), 3)))
replicas = cycle(["replica-a", "replica-b", "replica-c"])
print("round robin            :", [next(replicas) for _ in range(5)])
chunk_sizes = [300, 450, 200, 600, 150]
running = list(accumulate(chunk_sizes))
fits = sum(1 for total in running if total <= 1_000)
print("running token totals   :", running, "-> chunks that fit a 1,000-token budget:", fits)
print("pairwise gaps (ms)     :", [b - a for a, b in pairwise([0, 400, 1100, 1300])])
assert running == [300, 750, 950, 1550, 1700] and fits == 3

**`groupby` only groups *consecutive* items.** On unsorted input the same key shows up in several groups. Sort by the same key first — or use a `defaultdict(list)` when you don't need streaming.

**Predict, then run:** how many groups does the unsorted version produce?

In [ ]:
rows = [("small", 120), ("large", 900), ("small", 80), ("embed", 30), ("large", 400)]
unsorted_groups = [(model, [t for _, t in grp]) for model, grp in groupby(rows, key=lambda r: r[0])]
print("unsorted:", unsorted_groups)
by_model = sorted(rows, key=lambda r: r[0])
sorted_groups = {model: sum(t for _, t in grp) for model, grp in groupby(by_model, key=lambda r: r[0])}
print("sorted  :", sorted_groups)
assert len(unsorted_groups) == 5 and sorted_groups == {"embed": 30, "large": 1300, "small": 200}

## 9 · Streaming LLM tokens is a generator you loop over (`dt03`)

With `stream=True`, a chat API sends the answer as a sequence of small chunks while the model is still generating, and the SDK hands you an **iterator**:

```python
stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
for chunk in stream:                               # each chunk arrives as soon as it is generated
    delta = chunk.choices[0].delta.content or ""   # a few characters of text (None on some chunks)
    print(delta, end="", flush=True)
```

The fake below yields words with a small delay to show what that buys. The **total** time is about the same (the two runs differ a little because `time.sleep` is imprecise on a busy machine), but the **time to first token** — what the user actually feels — drops from "all of it" to "one token". This is *token* streaming; the CSV pipeline above is *data* streaming, a different meaning of the same word — exactly the ambiguity `dt03` warns about.

In [ ]:
import time

ANSWER = "Refunds are processed within five business days of receiving the returned item at our warehouse."
DELAY = 0.02

def fake_stream(text, delay=DELAY):
    for word in text.split():
        time.sleep(delay)                          # the "model" generating the next token
        yield word + " "

def fake_blocking(text, delay=DELAY):
    return "".join(fake_stream(text, delay))       # wait for everything, return once

def arrival_times(chunks):
    t0, times = time.perf_counter(), []
    for _ in chunks:
        times.append(time.perf_counter() - t0)
    return times

streamed = arrival_times(fake_stream(ANSWER))
t0 = time.perf_counter()
full_text = fake_blocking(ANSWER)
blocking_total = time.perf_counter() - t0
print(f"streaming: first word after {streamed[0] * 1000:.0f} ms, last after {streamed[-1] * 1000:.0f} ms")
print(f"blocking : first (and only) output after {blocking_total * 1000:.0f} ms")
assert full_text.split() == ANSWER.split() and streamed[0] < blocking_total / 5

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
n_words = len(streamed)
ax.step([0] + streamed, range(n_words + 1), where="post", label="streaming: words appear as they are generated")
ax.step([0, blocking_total], [0, n_words], where="post", label="blocking: everything arrives at the end")
ax.set_title("Same total time, very different time to first token")
ax.set_xlabel("seconds since the request"); ax.set_ylabel("words visible to the user"); ax.legend()
plt.show()

## 10 · Practical: batching texts for an embeddings API

Embedding endpoints accept a *list* of inputs per request, with limits on both the **number of items** and the **total tokens** per request. One text per request is slow and wasteful; everything at once gets rejected. `itertools.batched` handles the item limit; a small generator handles both limits at once. (A single item bigger than the size limit still goes out alone — chunk such documents first.)

In [ ]:
texts = [f"document {i}: " + "word " * (5 + (i * 7) % 40) for i in range(1_000)]
calls = 0
def fake_embed(batch):                     # stands in for client.embeddings.create(model=..., input=list(batch))
    global calls
    calls += 1
    return [[len(t), t.count(" ")] for t in batch]

vectors = []
for batch in batched(texts, 64):
    vectors.extend(fake_embed(batch))
print(f"{len(vectors)} vectors from {calls} calls (ceil(1000 / 64) = {-(-1000 // 64)})")
assert len(vectors) == 1000 and calls == 16

def pack_batches(items, max_items, max_chars):
    """Greedy batches that respect an item limit and a size limit (characters stand in for tokens)."""
    batch, size = [], 0
    for item in items:
        if batch and (len(batch) == max_items or size + len(item) > max_chars):
            yield batch
            batch, size = [], 0
        batch.append(item)
        size += len(item)
    if batch:
        yield batch

packed = list(pack_batches(texts, max_items=64, max_chars=8_000))
print(f"size-aware: {len(packed)} batches | largest has {max(map(len, packed))} items | "
      f"biggest payload {max(sum(map(len, b)) for b in packed):,} chars")
assert all(len(b) <= 64 and sum(map(len, b)) <= 8_000 for b in packed) and sum(map(len, packed)) == 1000

## Try it yourself

**1.** Rewrite as comprehensions: (a) the lengths of the words longer than 3 characters; (b) a dict from each word to its number of vowels.
```python
lengths = []
for w in words:
    if len(w) > 3:
        lengths.append(len(w))
```

In [ ]:
# Solution — try it yourself first, then run this
sample = ["rag", "vector", "index", "llm", "embedding"]
lengths = [len(w) for w in sample if len(w) > 3]
vowels = {w: sum(ch in "aeiou" for ch in w) for w in sample}
print(lengths, vowels)
assert lengths == [6, 5, 9] and vowels["embedding"] == 3

**2.** Write a generator `read_jsonl(lines)` that yields one parsed dict per line and silently skips blank or invalid lines — the shape of every log- and dataset-reading function.

In [ ]:
# Solution — try it yourself first, then run this
import json

def read_jsonl(lines):
    for line in lines:
        line = line.strip()
        if not line:
            continue
        try:
            yield json.loads(line)
        except json.JSONDecodeError:
            continue                     # in production: count and log these

raw_lines = ['{"q": "What is RAG?"}', "", "not json", '{"q": "What is BM25?"}']
records = list(read_jsonl(raw_lines))
print(records)
assert [r["q"] for r in records] == ["What is RAG?", "What is BM25?"]

**3.** Write `take(n, iterable)` with `islice` and `first(iterable, default=None)` with `next`. Both must work on an infinite iterator.

In [ ]:
# Solution — try it yourself first, then run this
def take(n, iterable):
    return list(islice(iterable, n))

def first(iterable, default=None):
    return next(iter(iterable), default)

print(take(3, count(10)), first(x for x in count() if x % 7 == 6), first([], "empty"))
assert take(3, count(10)) == [10, 11, 12] and first([], "empty") == "empty"

**4.** Write a generator `sliding_window(iterable, n)` that yields tuples of the last `n` items (use `collections.deque(maxlen=n)`), then use it for a moving average of request latencies.

In [ ]:
# Solution — try it yourself first, then run this
from collections import deque

def sliding_window(iterable, n):
    window = deque(maxlen=n)
    for item in iterable:
        window.append(item)
        if len(window) == n:
            yield tuple(window)

latencies_ms = [120, 180, 90, 400, 110, 130]
moving_avg = [sum(w) / len(w) for w in sliding_window(latencies_ms, 3)]
print([round(a, 1) for a in moving_avg])
assert list(sliding_window("abcd", 2)) == [("a", "b"), ("b", "c"), ("c", "d")] and len(moving_avg) == 4

## Say it in an interview

- **Generators (30 s, `dt04`):** "A function with `yield` returns a lazy iterator: each value is produced on demand, and the function pauses between values with its state intact. Memory is O(1) in the number of items, and generators compose into pipelines where each stage pulls from the previous one — the idiomatic answer to data bigger than memory."
- **The 5 GB CSV (`dt01`):** stream it — `for row in csv.DictReader(f)` and reduce inside the loop, so memory is bounded by the aggregate, not the file; or `pd.read_csv(chunksize=...)`, or DuckDB/Polars. Collecting the chunks into a list defeats the point.
- **Comprehension vs generator expression:** `[...]` builds the whole list (use it when you need `len`, indexing or several passes); `(...)` is lazy and single-use (use it for one pass: `sum`, `any`, `join`, a `for`).
- **Protocol:** an iterable has `__iter__`; an iterator has `__next__` and raises `StopIteration`; `for` calls `iter()` and then `next()` until `StopIteration`.
- **Traps:** generators, `map`, `zip` and files are single-use — a second pass is silently empty; `groupby` needs sorted input; `x in gen` consumes it.
- **Streaming (`dt03`):** say which one you mean. *Token* streaming improves time to first token, not total time or cost; *data* streaming bounds memory. Batching for embeddings respects both an item and a token limit per request.

## Next

- [08 · Classes, OOP and dataclasses](08_classes_oop_dataclasses.ipynb).
- [Chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) — windows with overlap, and better ways.
- [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb) — token streaming end to end.
- [Async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb) and [14 · async/await basics](14_async_await_basics.ipynb) — async generators build on this.
- Theory: the [python_basics course](../../python_basics/index.html) (chapter 8: comprehensions) · the [interview bank](../../ai_interview_prep/index.html) for `dt01`, `dt03`, `dt04`.